# XSe2 example

This notebook demonstrates a single UppASD simulation for XSe2 materials (X=Cr, Mn, Fe)

## Scientific objective
- Run a single finite-temperature simulation with controlled J1 + K
- Compute and visualize the resulting magnetic state from `restart._UppASD_.out`

## Learning goals
- Build and run the model with `SpinSystem`, `ExchangeShellTable`, and `AnisotropyTable`
- Tune simulation parameters to find the ground state
- Produce and analyze the magnetic structure with included plotting helpers


In [ ]:
%%capture
import os

# Install needed packages
!pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple uppasd
!pip install pyvista[jupyter]

os.environ["OMP_NUM_THREADS"] = "2"


In [ ]:
# Imports and basic setup (restored from reference notebook)
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyvista as pv

# Core UppASD classes for model setup, execution, and results parsing
from uppasd.core.system import SpinSystem
from uppasd.core.exchange import ExchangeShellTable
from uppasd.core.anisotropy import AnisotropyTable
from uppasd.input.inputdata import ASDInput
from uppasd.run.simulator import ASDWorkspace, UppASDSimulator
from uppasd.core.results import ASDResults
from uppasd.viz.io import timeseries

print("✅ Imports restored. Ensure you run cells that define `system`, `exchange`, and `inp` before preparing the workspace.")


## Step 1 — Define the crystal and initial magnetic state

We start from a minimal model to keep the physics and code path transparent.

### What each object means
- `cell`: lattice vectors that define periodic geometry
- `positions`: atom coordinates in the unit cell
- `species`: integer species labels (important for multi-component systems)
- `moments`: initial spin vectors per atom

Here we initialize moments along `+z`, which gives a simple reference state for observing relaxation behavior.

In [ ]:
# Set up a single unit cell with one atom
a = 1.0

# Hexagonal unit cell
#  1.0    0.0     0.0
# -0.5 sqrt(3)/2  0.0
#  0.0    0.0    sqrt(2)
cell = a * np.array([[1.0 , 0.0, 0.0], [-0.5 , np.sqrt(3.0)/2.0, 0.0],[0.0 , 0.0, np.sqrt(2.0)]])

# Positions in lattice coordinates
# 0.0 0.0 0.0
# 0.5 0.0 0.0
# 0.0 0.5 0.0
# 0.5 0.5 0.0
positions = np.array([[0.0 , 0.0, 0.0], [0.5 , 0.0, 0.0], [0.0 , 0.5, 0.0], [0.5 , 0.5, 0.0]])
# Transform to cartesian coordinates:
positions = positions @ cell.T
natom = positions.shape[0]
species = np.ones(natom, dtype=int)

# Define magnetic moments
m_FeSe2 = 3.593
m_MnSe2 = 3.685
m_CrSe2 = 2.753

# Setup moments
moments = np.zeros((natom, 3))
#moments[:, 2] = 1.0
moments[:, 2] = m_FeSe2

system = SpinSystem(cell, positions, species, moments)

## Step 2 — Define exchange interactions

The `ExchangeShellTable` encodes Heisenberg interactions shell-by-shell.

In the next code cell we define a frustrated J1 system with uniaxial anisotropy K. The exchange interactions are material specific and comes from the MIDB database https://aphys-midb.pdc.kth.se

The systems considered here are monolayers of $2H-XSe_2$ where $X=(Fe, Cr, Mn)$

In [ ]:
exchange = ExchangeShellTable()

# Definition of exchange interactions
J1_FeSe2 = -0.9621860632143219
J1_MnSe2 = -0.15209307331736713
J1_CrSe2 = -0.6404207391737375

# Exchange couplings J_ij in mRy (shell-resolved)
J_ij = [J1_FeSe2]
r_ij = np.array([[a/2.0, 0, 0]])

for ij, jij in enumerate(J_ij):
    if abs(jij)>0.0:
        exchange.add_bond(1, 1, ij+1, r_ij[ij], jij)

In [ ]:
# Step 2b — Define on-site anisotropy (kfile)
# Create an anisotropy table and add a uniaxial anisotropy on atom 1:
# Format: add_site(iatom, ani_type, K2, K4, direction)
# ani_type=1 for uniaxial, K2<0 for easy-axis

# Definition of anisotropies
K_FeSe2 = 0.1267115617468
K_MnSe2 = -0.286928923756659
K_CrSe2 = -0.094909458700267

# Set the anisotropy coupling
anisotropy = AnisotropyTable()
for i_site in range(natom):
    anisotropy.add_site(i_site + 1, 1, K_FeSe2, 0.0, [0.0, 0.0, 1.0])


## Step 3 — Configure input blocks

`ASDInput` organizes UppASD keywords by conceptual blocks (`system`, `initial`, `simulation`, `measurables`).

We use an LLG initial phase (`ip_mode='S'`) with scalar syntax:
- `ip_temp`: initial-phase temperature
- `ip_nstep`: number of initial-phase steps
- `ip_timestep`, `ip_damping`: integration controls

Then we set production simulation controls and request output needed for time-series analysis (`averages` and `totenergy`).

### Note:
In this particular example one needs to be cautions to relax/initialize the system carefully, since it is frustrated and anisotropic.

**Import & setup:** An imports/setup cell has been added above — run it first. 

If you have previously defined `system`, `exchange`, and `inp`, the workspace preparation cell will initialize the run directory.

In [ ]:
# Define simulation and measurement parameters (ASDInput)
inp = ASDInput()
Nx = 32
Ny = 32
Nz = 1

# Define a temperature to ensure same value is given in initial and measurement phases (optional)
Temperature = 0
Temperature_shift = 5.0

inp.block("system").set(
    ncell=(Nx, Ny, Nz),
    bc=("P", "P", "0"),
    do_prnstruct=2,
    sym=3,
    posfiletype='D',
 )

# Initial phase (LLG) using scalar syntax
inp.block("initial").set(
    ip_mode="H",
    ip_temp=Temperature+Temperature_shift,
    ip_mcnstep=1000,
    initmag=1,
 )

# Main simulation settings
inp.block("simulation").set(
    mode="S",
    nstep=10000,
    timestep=1e-15,
    damping=1.0,
    temperature=Temperature,
 )

# Request energy/averages/cumulant output
inp.block("measurables").set(plotenergy=1, do_avrg="Y", do_cumu="Y")

## Step 4 - Prepare workspace and run UppASD

The next two code cells do the full execution workflow:
1. Create a clean run directory and write all required input/interaction files
2. Initialize UppASD, run initial phase, perform measurement, and finalize

Keeping these steps explicit is useful for debugging and for teaching what happens under the hood.

### Note:
The `sim.initialize()` and `sim.finalize()` steps are always needed for any simulation.

In [ ]:
# Prepare workspace (will only run if system/inp/exchange are defined)
workdir = "XSe2_run"
if 'system' in globals() and 'inp' in globals() and 'exchange' in globals():
    ws = ASDWorkspace(workdir, clean=True)
    # Pass anisotropy using the `ani=` keyword (alias for anisotropy)
    ws.prepare(system=system, inp=inp, exchange=exchange, ani=anisotropy)
    print(f'Workspace prepared at {workdir}')
else:
    print("Variables 'system', 'inp', or 'exchange' not defined. Define them before preparing workspace.")

In [ ]:
sim = UppASDSimulator(ws)
sim.initialize()
sim.run_init_phase()
sim.measure()
sim.finalize()

## Step 5 - Gather results and visualize

After a simulations, the output data needs to be gathered which is done using the `ASDResults` class.

Here we then plot the final magnetic structure from the `restart.simid.out` file.


In [ ]:
# Post-processing for a single run: magnetization and energy vs time
results = ASDResults(workdir)
print("Available tables:", results.available)

In order to make the data formats compatible with the plotting cell, the `ASDResults` data is converted to `numpy` arrays.

In [ ]:

coord = results.coord
restart = results.restart
if coord is None or restart is None:
    raise RuntimeError("Missing coord or restart in results")

# Exact map: (replica, iatom) -> index in coord['raw']
coord_pairs = [(int(r), int(i)) for r, i in zip(coord["replica"], coord["iatom"])]
coord_map = {pair: idx for idx, pair in enumerate(coord_pairs)}

def find_coord_index(replica, iatom):
    key = (int(replica), int(iatom))
    if key in coord_map:
        return coord_map[key]
    # fallback: any coord row with same iatom
    matches = np.where(coord["iatom"] == int(iatom))[0]
    if matches.size == 1:
        return matches[0]
    if matches.size > 1:
        # prefer same replica if present among matches
        for m in matches:
            if int(coord["replica"][m]) == int(replica):
                return m
        return matches[0]  # ambiguous: pick first
    # nothing found -> raise informative error
    available = sorted(set(coord_pairs))
    raise KeyError(
        f"No coord entry for replica/iatom {key}. "
        f"Available (replica, iatom) pairs (sample): {available[:20]}"
    )

# Build arrays aligned to restart ordering
n = len(restart["iatom"])
pos = np.empty((n, 3), dtype=float)
vecs = np.empty((n, 3), dtype=float)

for i in range(n):
    repl = restart["replica"][i]
    iatom = restart["iatom"][i]
    idx = find_coord_index(repl, iatom)
    pos[i, :] = (coord["x"][idx], coord["y"][idx], coord["z"][idx])
    mag = restart["magnitude"][i]
    vecs[i, :] = np.array((restart["mx"][i], restart["my"][i], restart["mz"][i])) * mag

# print("pos.shape =", pos.shape, "vecs.shape =", vecs.shape)

With the positions and moments in proper format, we can visualize the magnetic structure interactively using `pyvista` as below.

Here you can try to vary the plotting parameters such as
- glyphs : Cone, Sphere, Arrow
- coloring component: Mx, My, Mz

but also different rendering options and glyph resolutions.


In [ ]:
# Set environment variable for trame in Jupyter to help with Colab proxying
os.environ["TRIDENT_VIEWER"] = "jupyter"
pv.set_jupyter_backend('html')


# 2. Create the PyVista PolyData object
mesh = pv.PolyData(pos)
mesh['spins'] = vecs
mesh['Mx'] = vecs[:, 0]
mesh['My'] = vecs[:, 1]
mesh['Mz'] = vecs[:, 2]

# 3. Create High-Resolution Geometry
# Arrow
arrow_geom = pv.Arrow(tip_radius=0.25, tip_length=0.4, shaft_radius=0.1, shaft_resolution=26, tip_resolution=32)
# Cone
# arrow_geom = pv.Cone(radius=0.5, height=1.0, resolution=32)
# Sphere
# arrow_geom = pv.Sphere(radius=0.45, phi_resolution=12, theta_resolution=12)

# Glyphing
arrows = mesh.glyph(orient='spins', scale=False, factor=0.5, geom=arrow_geom)

# 4. Interactive Plotting with Advanced Shading
plotter = pv.Plotter(notebook=True)

# Add the spins with Gouraud shading and Specular highlights
plotter.add_mesh(
    arrows,
    scalars='Mz',
    cmap='RdBu',
    #clim=[-1, 1],
    show_scalar_bar=False,   # ← suppress colorbar
    lighting=True,
    smooth_shading=True,   # Enables Gouraud shading (smooths facets)
    specular=0.6,          # Adds a "shiny" reflection point
    specular_power=30,     # Sharpness of the shine (higher = more metallic)
    ambient=0.2,            # Ensures shadows aren't pitch black
    pbr=True,
    metallic=0.2,
    roughness=0.6,
)

# Optional: Add the atomic lattice with Gouraud shading as well
#lattice = mesh.glyph(geom=pv.Sphere(radius=0.05, phi_resolution=20, theta_resolution=20))
#plotter.add_mesh(lattice, color='grey', opacity=0.4, smooth_shading=True)

# Add SSAO for improved look
#plotter.enable_ssao(radius=0.6, bias=0.1)

# Post-processing:
plotter.enable_eye_dome_lighting()
plotter.enable_depth_of_field()
## Adjust focal point to the center of your spins
plotter.camera.focal_point = arrows.center

# Final Camera Setup
plotter.camera_position = 'iso'
plotter.reset_camera()

plotter.background_color = "white"
plotter.show()

To make a snapshot for downloading, we can run the cell below. 

Note: Here we set the perspective programatically so the image will not be the same as your viewport above. But you can try to change the camera position if wanted.

In [ ]:
# Run this cell for snapshot of the magnetization
plotter.camera_position = 'xy'
plotter.reset_camera()
center = arrows.center
bounds = arrows.bounds

size = max(
    bounds[1]-bounds[0],
    bounds[3]-bounds[2],
    bounds[5]-bounds[4]
)

plotter.camera.focal_point = center
plotter.camera.position = (
    center[0],
    center[1],
    center[2] + 1*size
)
plotter.screenshot(filename="spins.png",return_img=False,window_size=[1024,768])

We can also print the energy evolution and magnetization to track the relaxation dynamics.

In [ ]:
# Extract energy and magnetization from output dictionaries
energy_data_k = results['totenergy']
time_k = energy_data_k['iter']
energy_k = energy_data_k['tot']

avg_data_k = results['averages']
time_avg = avg_data_k['iter']
mag_k = np.column_stack([avg_data_k['mx'], avg_data_k['my'], avg_data_k['mz']])

print(f"Final state:")
print(f"   Final energy: {energy_k[-1]:.4f} mRy/atom")
print(f"   Final <M>: ({avg_data_k['mx'][-1]:.4f}, {avg_data_k['my'][-1]:.4f}, {avg_data_k['mz'][-1]:.4f})")

# Plot energy and magnetization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(time_k, energy_k, linewidth=1.5, color='darkgreen')
axes[0].set_xlabel('Iteration', fontsize=11)
axes[0].set_ylabel('Total Energy', fontsize=11)
axes[0].set_title('XSe2 system: Energy vs Time', fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_avg, avg_data_k['mx'], label='Mx', linewidth=1.5)
axes[1].plot(time_avg, avg_data_k['my'], label='My', linewidth=1.5)
axes[1].plot(time_avg, avg_data_k['mz'], label='Mz', linewidth=1.5)
axes[1].set_xlabel('Iteration', fontsize=11)
axes[1].set_ylabel('Magnetization (per atom)', fontsize=11)
axes[1].set_title('XSe2 system: Magnetization vs Time', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
